# M5v2 v2 Kaggle Training (T4)

**목표**: mAP50 0.626 → **0.85+ 도전**

**환경**: Kaggle Notebooks T4 GPU (12h 한도, 30h/주)

**Dataset 준비:**
- Kaggle Datasets에 `frames_ade.zip` 업로드 → 본인 dataset slug 확인
- 우측 사이드바 **+ Add data** → 본인 frames-ade 데이터셋 추가

**예상 시간**: 7~9시간

**런타임**: GPU T4 x2 또는 GPU P100

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
!pip install -q ultralytics onnx onnxslim onnxruntime-gpu numpy pillow

In [ ]:
import os, shutil, zipfile
from pathlib import Path

# ★★★ 본인이 업로드한 Kaggle Dataset 경로 ★★★
# 우측 사이드바 'Input' 섹션에서 데이터셋 폴더 이름 확인
# 보통: /kaggle/input/{본인이 dataset 만들때 입력한 slug}/
INPUT_DIR = Path('/kaggle/input/frames-ade')  # ← 본인 dataset slug로 변경 필요
ZIP_NAME = 'frames_ade.zip'

WORK = Path('/kaggle/working/m5v2_v2')
WORK.mkdir(parents=True, exist_ok=True)

# 데이터셋 압축 해제
zip_path = INPUT_DIR / ZIP_NAME
print(f'zip exists? {zip_path.exists()} -> {zip_path}')
assert zip_path.exists(), f'{zip_path} 없음. INPUT_DIR을 본인 dataset 경로로 수정 필요'

ds_dir = WORK / 'frames_ade'
if not (ds_dir / 'images' / 'val').exists():
    if ds_dir.exists():
        shutil.rmtree(ds_dir, ignore_errors=True)
    print('Extracting frames_ade.zip ...')
    with zipfile.ZipFile(zip_path) as z:
        for m in z.namelist():
            rel = m.replace('\\', '/')
            target = WORK / rel
            if rel.endswith('/'):
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(m) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
    print(f'Done: {ds_dir}')
else:
    print(f'Already extracted: {ds_dir}')

for split in ['train', 'val', 'test']:
    p = ds_dir / 'images' / split
    if p.exists():
        print(f'  {split}: {len(list(p.glob("*")))} images')

In [ ]:
# data.yaml 생성
yaml_text = f"""path: {ds_dir}
train: images/train
val: images/val
test: images/test

nc: 4
names:
  0: wall_edge
  1: ceiling_edge
  2: door_frame
  3: window_frame
"""
data_yaml = WORK / 'frames_ade.yaml'
data_yaml.write_text(yaml_text)
print(data_yaml.read_text())

## 학습 (yolov11m + 1280 + 50ep)

Kaggle은 `/kaggle/working/`에 저장하면 노트북 종료 시 자동으로 outputs에 저장 → 다운로드 가능.

In [ ]:
from ultralytics import YOLO

PROJECT = WORK / 'runs'
PROJECT.mkdir(parents=True, exist_ok=True)

model = YOLO('yolo11m.pt')

results = model.train(
    data=str(data_yaml),
    epochs=50,
    batch=4,                # T4 16GB + 1280 → batch=4
    imgsz=1280,
    cache='disk',
    workers=4,
    optimizer='AdamW',
    lr0=5e-4,
    lrf=0.01,
    cos_lr=True,
    patience=15,
    warmup_epochs=3,
    close_mosaic=15,
    freeze=0,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
    degrees=3.0, translate=0.1, scale=0.3,
    shear=1.0, perspective=0.0005,
    flipud=0.0, fliplr=0.5,
    mosaic=1.0, mixup=0.1, copy_paste=0.3,
    erasing=0.0,
    multi_scale=0.2,
    save_period=5,
    plots=True,
    project=str(PROJECT),
    name='m5v2_v2',
    exist_ok=True,
)

print('Train done.')

## ONNX Export + 평가 + 결과 저장

In [ ]:
best_path = PROJECT / 'm5v2_v2' / 'weights' / 'best.pt'
print(f'best.pt: {best_path} ({best_path.stat().st_size/1024/1024:.1f}MB)')

best_model = YOLO(str(best_path))
best_model.export(format='onnx', opset=17, dynamic=True, simplify=True)
onnx_path = best_path.with_suffix('.onnx')
print(f'ONNX: {onnx_path} ({onnx_path.stat().st_size/1024/1024:.1f}MB)')

metrics = best_model.val(data=str(data_yaml), imgsz=1280, batch=4)
print(f'\n=== M5v2 v2 결과 ===')
print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
print(f'precision: {metrics.box.mp:.4f}')
print(f'recall:    {metrics.box.mr:.4f}')
print(f'0.9? {"YES ✅" if metrics.box.map50 >= 0.9 else "NO"}')

In [ ]:
# 결과 파일을 /kaggle/working 루트로 복사 (다운로드 쉽게)
OUT = Path('/kaggle/working/m5v2_v2_results')
OUT.mkdir(parents=True, exist_ok=True)

shutil.copy2(onnx_path, OUT / 'm5_yolo_seg_frames.onnx')
shutil.copy2(best_path, OUT / 'm5v2_v2_best.pt')

results_csv = best_path.parent.parent / 'results.csv'
if results_csv.exists():
    shutil.copy2(results_csv, OUT / 'results.csv')

for plot in best_path.parent.parent.glob('*.png'):
    shutil.copy2(plot, OUT / plot.name)

print('Saved to:', OUT)
for f in OUT.iterdir():
    print(f'  {f.name} ({f.stat().st_size/1024/1024:.1f}MB)')

print('\n노트북 우측 패널 → "Output" 탭에서 다운로드 가능')